# Agents & Custom Tools — ReAct, Tool Routing & Custom Functions

Give your LLM the ability to take actions — run calculations, call APIs, and decide which tool to use.

---

In [ ]:
!pip install -q langchain langchain-openai

---

## 1 · Agents vs Chains

**The Problem** — Chains follow fixed pipelines. Agents let the LLM decide which tools to use based on the input.

**The Solution** — An agent uses the LLM as a reasoning engine. It thinks, picks a tool, observes the result, and decides what to do next.

> A chain is a recipe you follow step by step. An agent is a chef who reads the order and decides which tools to use.

---

## 2 · Custom Tools with @tool

**The Problem** — You need the agent to call your own functions.

**The Solution** — The `@tool` decorator converts any Python function into a LangChain tool. The agent uses the function name, docstring, and type hints to decide when to call it.

> The docstring is the tool's instruction manual. A vague docstring = the agent won't know when to use it.

In [ ]:
from langchain.tools import tool

# ── Define custom tools ──
# Each tool is a plain Python function wrapped with @tool
# The docstring tells the agent WHEN to use this tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together. Use this when you need to calculate a product."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Add two integers together. Use this when you need to calculate a sum."""
    return a + b

@tool
def word_count(text: str) -> int:
    """Count the number of words in a given text string."""
    return len(text.split())

# ── Inspect the tool metadata — this is what the agent sees ──
for t in [multiply, add, word_count]:
    print(f"Name: {t.name}")
    print(f"Description: {t.description}")
    print(f"Args: {t.args}")
    print()

In [ ]:
# ── Call tools directly (for testing) ──
print(f"multiply(6, 7) = {multiply.invoke({'a': 6, 'b': 7})}")
print(f"add(10, 20) = {add.invoke({'a': 10, 'b': 20})}")
print(f"word_count('hello world foo') = {word_count.invoke({'text': 'hello world foo'})}")

### What Just Happened?

- `@tool` converted plain Python functions into LangChain `Tool` objects
- Each tool exposes `name`, `description`, and `args` — the agent reads these to decide which tool to call
- Tools can be called directly via `.invoke()` for testing

---

## 3 · Tool-Calling Agent — The Modern Approach

**The Problem** — You need an agent that picks and calls tools reliably.

**The Solution** — `create_tool_calling_agent` uses the LLM's native function-calling capability (structured JSON outputs) instead of text-based parsing.

> The LLM doesn't generate text instructions to call tools — it returns structured function calls directly. More reliable, fewer parsing errors.

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ── LLM with tool-calling support ──
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── Tools ──
tools = [multiply, add, word_count]

# ── Prompt with required placeholders ──
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools when needed to answer questions accurately."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),  # required: agent's working memory
])

# ── Create the agent ──
agent = create_tool_calling_agent(llm, tools, prompt)

# ── Wrap in AgentExecutor for error handling and iteration control ──
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,                  # shows reasoning steps
    max_iterations=5,              # prevent infinite loops
    handle_parsing_errors=True,    # graceful error recovery
)

print("Agent ready with tools:", [t.name for t in tools])

In [ ]:
# ── Test: simple calculation ──
result = executor.invoke({"input": "What is 42 multiplied by 17?"})
print(f"\nAnswer: {result['output']}")

In [ ]:
# ── Test: tool routing — the agent picks the right tool ──
result = executor.invoke({"input": "How many words are in the sentence 'The quick brown fox jumps over the lazy dog'?"})
print(f"\nAnswer: {result['output']}")

In [ ]:
# ── Test: multi-step — agent uses multiple tools ──
result = executor.invoke({"input": "What is 15 + 27, and then multiply that result by 3?"})
print(f"\nAnswer: {result['output']}")

In [ ]:
# ── Test: no tool needed — agent answers directly ──
result = executor.invoke({"input": "What is the capital of France?"})
print(f"\nAnswer: {result['output']}")

### What Just Happened?

- The agent decided **which tool** to call based on the question
- For "multiply" questions, it picked `multiply`. For word counting, it picked `word_count`
- For multi-step tasks, it chained tools: `add` → `multiply`
- For general knowledge questions, it answered directly without tools
- `verbose=True` showed the full Thought → Action → Observation loop

---

## 4 · Structured Tool Inputs with Pydantic

**The Problem** — Complex tools need validated, multi-field inputs.

**The Solution** — Use Pydantic models to define tool input schemas.

> Type hints tell the agent what to pass in. Pydantic validates it before execution.

In [ ]:
from pydantic import BaseModel, Field
from langchain.tools import tool

# ── Define a structured input schema ──
class TemperatureConversion(BaseModel):
    value: float = Field(description="The temperature value to convert")
    from_unit: str = Field(description="Source unit: 'celsius' or 'fahrenheit'")
    to_unit: str = Field(description="Target unit: 'celsius' or 'fahrenheit'")

@tool(args_schema=TemperatureConversion)
def convert_temperature(value: float, from_unit: str, to_unit: str) -> str:
    """Convert a temperature between Celsius and Fahrenheit."""
    if from_unit == "celsius" and to_unit == "fahrenheit":
        result = (value * 9/5) + 32
    elif from_unit == "fahrenheit" and to_unit == "celsius":
        result = (value - 32) * 5/9
    else:
        return f"Cannot convert from {from_unit} to {to_unit}"
    return f"{value}° {from_unit.title()} = {result:.1f}° {to_unit.title()}"

# ── Inspect the schema the agent will see ──
print(f"Name: {convert_temperature.name}")
print(f"Description: {convert_temperature.description}")
print(f"Args schema: {convert_temperature.args}")

In [ ]:
# ── Create an agent with the structured tool ──
tools_extended = [multiply, add, word_count, convert_temperature]

agent_ext = create_tool_calling_agent(llm, tools_extended, prompt)
executor_ext = AgentExecutor(
    agent=agent_ext, tools=tools_extended,
    verbose=True, max_iterations=5, handle_parsing_errors=True
)

result = executor_ext.invoke({"input": "What is 100 degrees Fahrenheit in Celsius?"})
print(f"\nAnswer: {result['output']}")

### What Just Happened?

- The Pydantic schema told the agent exactly what fields to provide
- The agent correctly populated `value`, `from_unit`, and `to_unit`
- Input validation happens automatically before the tool executes

---

## 5 · Real-World Tool — Mock API Integration

Tools can call APIs, query databases, or interact with any external system.

In [ ]:
import json
from langchain.tools import tool

# ── Simulate an API-backed tool ──
# In production, this would call a real API

FAKE_DB = {
    "AAPL": {"price": 189.50, "change": "+1.2%", "name": "Apple Inc."},
    "GOOGL": {"price": 141.80, "change": "-0.5%", "name": "Alphabet Inc."},
    "MSFT": {"price": 378.90, "change": "+0.8%", "name": "Microsoft Corp."},
    "TSLA": {"price": 245.60, "change": "+2.1%", "name": "Tesla Inc."},
}

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price and daily change for a given stock ticker symbol (e.g., AAPL, GOOGL, MSFT)."""
    ticker = ticker.upper()
    if ticker in FAKE_DB:
        data = FAKE_DB[ticker]
        return f"{data['name']} ({ticker}): ${data['price']} ({data['change']})"
    return f"Stock ticker '{ticker}' not found."

@tool
def compare_stocks(ticker1: str, ticker2: str) -> str:
    """Compare the prices of two stocks by their ticker symbols."""
    t1, t2 = ticker1.upper(), ticker2.upper()
    if t1 in FAKE_DB and t2 in FAKE_DB:
        d1, d2 = FAKE_DB[t1], FAKE_DB[t2]
        diff = d1['price'] - d2['price']
        return (f"{d1['name']}: ${d1['price']} vs {d2['name']}: ${d2['price']}. "
                f"Difference: ${abs(diff):.2f} ({t1 if diff > 0 else t2} is higher)")
    return "One or both tickers not found."

# ── Agent with stock tools ──
stock_tools = [get_stock_price, compare_stocks, multiply, add]
agent_stocks = create_tool_calling_agent(llm, stock_tools, prompt)
executor_stocks = AgentExecutor(
    agent=agent_stocks, tools=stock_tools,
    verbose=True, max_iterations=5, handle_parsing_errors=True
)

print("Stock agent ready.")

In [ ]:
# ── Single tool call ──
result = executor_stocks.invoke({"input": "What's the current price of Apple stock?"})
print(f"\nAnswer: {result['output']}")

In [ ]:
# ── Multi-tool routing ──
result = executor_stocks.invoke({"input": "Compare Apple and Google stock prices, then tell me what their combined price would be."})
print(f"\nAnswer: {result['output']}")

### What Just Happened?

- The agent routed questions to the appropriate tools automatically
- For "compare and add," it chained `compare_stocks` → `add` in a single conversation
- In production, `get_stock_price` would call a real API (Alpha Vantage, Yahoo Finance, etc.)

---

## 6 · Error Handling and Edge Cases

In [ ]:
# ── Tool not found / invalid input ──
result = executor_stocks.invoke({"input": "What's the stock price of ZZZZ?"})
print(f"\nAnswer: {result['output']}")
print("\n→ The agent handled the 'not found' gracefully.")

In [ ]:
# ── No tool needed ──
result = executor_stocks.invoke({"input": "Tell me a joke."})
print(f"\nAnswer: {result['output']}")
print("\n→ The agent correctly skipped tools and answered directly.")

---

## Key Takeaways

| Component | What It Does | Key Detail |
|-----------|-------------|------------|
| `@tool` | Wraps a function as a tool | Docstring = when the agent uses it |
| `create_tool_calling_agent` | Creates agent with native function calling | Most reliable for modern LLMs |
| `AgentExecutor` | Runs the agent loop | Error handling, max iterations |
| `args_schema` | Pydantic validation for inputs | Structured multi-field tools |
| `verbose=True` | Shows reasoning steps | Essential for debugging |
| `max_iterations` | Prevents infinite loops | Default: 15, set lower for safety |

**Practical advice:** Use `create_tool_calling_agent` for modern LLMs (GPT-4o, Claude). Write clear, specific docstrings for every tool. Always set `max_iterations` and `handle_parsing_errors=True`.

**Next →** [10 · Callbacks & Tracing](../10-callbacks-tracing/)

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*